# Four energy drink flavours: Friedman test finds no winner

> **Before you run:** choose **Copy to Drive** at the top of this page (or **File → Save a copy in Drive**). Your copy opens in a new tab. In your copy, choose **Runtime → Run all**. Your copy is yours, so Colab runs it without a warning, and it keeps your changes.

**Friedman Test** · a worked example from the StatNex teaching guide

**The story.** Eight tasters each scored four flavours of a new energy drink from 0 to 100. Some tasters are generous and score everything high (taster 6 gives 73 to 98), others are harsh (taster 7 gives 24 to 50). The company can launch only one new flavour this year.

**The question.** Is there a flavour that tasters consistently prefer, once we allow for some tasters being harsher than others?

If you run this page without a copy, Colab first warns that the notebook was not written by Google. It reads none of your data, so **Run anyway** is safe, but your changes are not kept.

The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## The table

This cell saves the table `06_nonparametric/friedman_test.csv` (8 rows) as `data.csv`, the file the code reads.

In [ ]:
with open("data.csv", "w", encoding="utf-8") as file:
    file.write("""\
Taster,Original,Mango,Berry,Mint
1.0,66.0,74.0,59.0,45.0
2.0,66.0,55.0,57.0,73.0
3.0,53.0,74.0,52.0,44.0
4.0,63.0,45.0,66.0,75.0
5.0,60.0,68.0,57.0,40.0
6.0,84.0,73.0,91.0,98.0
7.0,36.0,50.0,43.0,24.0
8.0,65.0,45.0,63.0,73.0
""")

## Libraries

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

pd.set_option("display.width", 120)

## The data

In [ ]:
# One row per case (subject), one column per condition. Only rows with a number in every
# condition can be ranked, so a row with a blank or text cell is dropped.
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
levels = ["Original", "Mango", "Berry", "Mint"]
values = data[levels].apply(pd.to_numeric, errors="coerce")
complete = values.dropna()
n, k = complete.shape
total = len(values)
cases = pd.DataFrame({"n": [n, total - n, total], "%": [n / total, (total - n) / total, 1.0]},
                     index=["valid cases", "incomplete cases", "total"])
print(cases)

## Mean rank

In [ ]:
# Values are ranked within each row (ties share the average rank), so a case that
# scores high everywhere counts the same as one that scores low everywhere.
ranks = complete.rank(axis=1)
mean_ranks = ranks.mean()
print()
print(mean_ranks.to_frame("mean rank").T)

## Friedman test

In [ ]:
# If the conditions were all alike, each case's ranks would fall in a random order and
# the rank sums would be about equal; chi-square grows as they spread apart, and is read
# on k - 1 df. This first one is the statistic without a tie correction, written out
# because scipy.stats.friedmanchisquare (and R's friedman.test) only give the corrected
# one, which differs when a row has ties.
rank_sums = ranks.sum().to_numpy()
chi2 = 12.0 / (n * k * (k + 1)) * (rank_sums**2).sum() - 3 * n * (k + 1)
df = k - 1
p = stats.chi2.sf(chi2, df)
print(f"\nchi-square = {chi2:.6f}, df = {df}, P = {p:.6g}")
# When a case has tied values, the app adds the tie-corrected statistic: chi-square
# divided by 1 - sum(t^3 - t) / (n k (k^2 - 1)), t the size of each tied group. That is
# scipy.stats.friedmanchisquare's value, and the one to report when there are ties.
tie_term = 0.0
for row in ranks.to_numpy():
    _, sizes = np.unique(row, return_counts=True)
    tie_term += (sizes**3 - sizes).sum()
# Written out because friedmanchisquare refuses two conditions, which the test allows.
chi2_corrected = chi2 / (1 - tie_term / (n * k * (k * k - 1)))
if tie_term > 0:
    p_corrected = stats.chi2.sf(chi2_corrected, df)
    print(f"corrected for ties: chi-square = {chi2_corrected:.6f}, P = {p_corrected:.6g}")

## Kendall's coefficient of concordance

In [ ]:
# W is the same statistic on a 0-1 scale: 1 when every case ranks the conditions alike.
# With ties it takes the tie-corrected statistic (Kendall and Gibbons), so that complete
# agreement still gives 1.
w = chi2_corrected / (n * (k - 1))
print(f"W = {w:.6f}")

## What the result says

The Friedman test compares several items scored by the same people. It ranks the four flavours inside each taster, so a harsh taster and a generous taster count the same.

The mean ranks are 2.625 for Original, 2.5 for Mango, 2.375 for Berry and 2.5 for Mint: all almost exactly in the middle. The chi-square statistic is 0.15 with 3 degrees of freedom, and the P value is 0.985. The P value is the chance of differences this large if no flavour were preferred, so there is no sign of a favourite. Kendall's coefficient of concordance, which measures how much the tasters agree, is 0.006, close to no agreement.

No flavour wins across the 8 tasters. This does not show that the flavours are liked equally: the tasters may simply disagree with each other. The company cannot pick a flavour from this result.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.